# AI Use Case 5 — Pipeline completo: triagem de tickets com LLM

Juntando as três ferramentas, cada uma no papel em que é melhor:

| camada | ferramenta | papel |
|---|---|---|
| configuração | **Pydantic** (`pydantic-settings`) | modelos, limiares e limites vindos de env/arquivo, validados no startup |
| entrada da API | **Pydantic** | `TicketIn` valida o que o cliente manda (422 se inválido) |
| vocabulário | **Enum** | `Intent`, `Priority`, `Team` — prompt, schema, roteamento e métricas |
| saída do LLM | **Pydantic** | `TriageDecision` como structured output + validators semânticos + retry |
| estado interno | **dataclass** | `TriageRun` (tentativas, uso, eventos) — rápido, sem validação repetida |
| resposta da API | **Pydantic** | `TicketOut` controla exatamente o que sai |

```mermaid
flowchart LR
    A["POST /tickets<br/>JSON"] --> B["TicketIn<br/>(Pydantic)"]
    B -->|422| X["erro ao cliente"]
    B --> C["prompt gerado<br/>dos Enums + schema"]
    C --> D["LLM"]
    D --> E{"TriageDecision<br/>valida?"}
    E -->|"não (feedback)"| D
    E -->|sim| F["regras de negócio<br/>Intent→Team, escalonamento<br/>(Enums)"]
    F --> G["TriageRun<br/>(dataclass: uso, eventos)"]
    G --> H["TicketOut<br/>(Pydantic)"]
    H --> I["201 JSON"]
```

In [1]:
import json, os, re, time
from dataclasses import dataclass, field
from enum import StrEnum
from typing import Literal
from pydantic import BaseModel, ConfigDict, EmailStr, Field, ValidationError, model_validator
from pydantic_settings import BaseSettings, SettingsConfigDict
from fake_llm import ScriptedLLM, LLMResponse

## 1. Configuração (Pydantic Settings)

In [2]:
class TriageSettings(BaseSettings):
    model_config = SettingsConfigDict(env_prefix="TRIAGE_", frozen=True)

    model: str = "medium-model"
    max_attempts: int = Field(default=3, ge=1, le=5)
    min_confidence: float = Field(default=0.7, ge=0, le=1)
    vip_domains: tuple[str, ...] = ("bigcorp.com",)

os.environ["TRIAGE_MIN_CONFIDENCE"] = "0.75"
os.environ["TRIAGE_VIP_DOMAINS"] = '["bigcorp.com", "partner.io"]'
settings = TriageSettings()
print(settings)

model='medium-model' max_attempts=3 min_confidence=0.75 vip_domains=('bigcorp.com', 'partner.io')


## 2. Vocabulário (Enums)

In [3]:
class Intent(StrEnum):
    def __new__(cls, value: str, description: str):
        obj = str.__new__(cls, value)
        obj._value_, obj.description = value, description
        return obj

    BILLING = ("billing", "cobrança, fatura, reembolso, pagamento")
    BUG = ("bug", "erro, falha, algo quebrado no produto")
    ACCOUNT = ("account", "login, senha, acesso, dados cadastrais")
    FEATURE = ("feature", "pedido de funcionalidade nova ou melhoria")
    OTHER = ("other", "nenhuma das anteriores")

class Priority(StrEnum):
    """StrEnum (não IntEnum): a API emite "P2", legível, e não se confunde com contagens."""
    P1 = "P1"      # sistema parado / perda financeira
    P2 = "P2"
    P3 = "P3"
    P4 = "P4"      # dúvida simples

    @property
    def rank(self) -> int:            # 1 = mais urgente
        return int(self.value[1])

    def bump(self) -> "Priority":
        return Priority(f"P{max(self.rank - 1, 1)}")

class Team(StrEnum):
    FINANCE = "finance"
    ENGINEERING = "engineering"
    SUPPORT = "support"
    PRODUCT = "product"
    HUMAN_TRIAGE = "human_triage"

ROUTING: dict[Intent, Team] = {
    Intent.BILLING: Team.FINANCE, Intent.BUG: Team.ENGINEERING, Intent.ACCOUNT: Team.SUPPORT,
    Intent.FEATURE: Team.PRODUCT, Intent.OTHER: Team.SUPPORT,
}
assert set(ROUTING) == set(Intent)

## 3. Contratos da borda (Pydantic)

In [4]:
class TicketIn(BaseModel):
    model_config = ConfigDict(extra="forbid", str_strip_whitespace=True)
    customer_email: EmailStr
    subject: str = Field(min_length=3, max_length=200)
    body: str = Field(min_length=10, max_length=5_000)

class TriageDecision(BaseModel):
    """Saída esperada do LLM."""
    model_config = ConfigDict(extra="forbid")
    intent: Intent
    priority: Priority = Field(description="P1 = sistema parado ou perda financeira; P4 = dúvida simples")
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=10, max_length=160, description="Resumo em uma frase, em português")
    order_id: str | None = Field(description="Número do pedido citado no texto; null se não houver")
    sentiment: Literal["calm", "frustrated", "angry"]

    @model_validator(mode="after")
    def p1_needs_justification(self):
        if self.priority is Priority.P1 and self.intent in (Intent.FEATURE, Intent.OTHER):
            raise ValueError("P1 só é permitido para billing, bug ou account")
        return self

class TicketOut(BaseModel):
    ticket_id: str
    team: Team
    priority: Priority
    intent: Intent
    summary: str
    needs_human_review: bool
    order_id: str | None
    llm_attempts: int

## 4. Estado interno (dataclass)

In [5]:
@dataclass(slots=True)
class LLMCallRecord:
    attempt: int
    ok: bool
    input_tokens: int
    output_tokens: int
    error: str | None = None

@dataclass(slots=True)
class TriageRun:
    ticket_id: str
    started: float = field(default_factory=time.perf_counter)
    calls: list[LLMCallRecord] = field(default_factory=list)
    events: list[str] = field(default_factory=list)

    @property
    def tokens(self) -> int:
        return sum(c.input_tokens + c.output_tokens for c in self.calls)

    @property
    def elapsed_ms(self) -> float:
        return (time.perf_counter() - self.started) * 1000

    def log(self, msg: str) -> None:
        self.events.append(msg)

## 5. O pipeline

In [6]:
SYSTEM_PROMPT = (
    "Você faz triagem de tickets de suporte. Intenções possíveis:\n"
    + "\n".join(f"- {i.value}: {i.description}" for i in Intent)
    + "\nResponda APENAS com JSON neste schema:\n"
    + json.dumps(TriageDecision.model_json_schema(), ensure_ascii=False)
)

def extract_json(text: str) -> str:
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        raise ValueError("resposta sem JSON")
    return m.group(0)

def call_llm_structured(llm, ticket: TicketIn, run: TriageRun) -> TriageDecision:
    messages = [{"role": "user", "content": f"Assunto: {ticket.subject}\n\n{ticket.body}"}]
    for attempt in range(1, settings.max_attempts + 1):
        resp = llm(messages, system=SYSTEM_PROMPT, model=settings.model)
        try:
            decision = TriageDecision.model_validate_json(extract_json(resp.text))
            run.calls.append(LLMCallRecord(attempt, True, resp.input_tokens, resp.output_tokens))
            return decision
        except (ValidationError, ValueError) as e:
            err = "; ".join(f"{'.'.join(map(str, x['loc'])) or '(objeto)'}: {x['msg']}" for x in e.errors(include_url=False)) \
                if isinstance(e, ValidationError) else str(e)
            run.calls.append(LLMCallRecord(attempt, False, resp.input_tokens, resp.output_tokens, err))
            run.log(f"tentativa {attempt} inválida: {err}")
            messages += [{"role": "assistant", "content": resp.text},
                         {"role": "user", "content": f"Corrija o JSON. Erros: {err}"}]
    raise RuntimeError("LLM não produziu triagem válida")

def apply_business_rules(ticket: TicketIn, d: TriageDecision, run: TriageRun) -> tuple[Team, Priority, bool]:
    team = ROUTING[d.intent]
    priority = d.priority
    needs_review = False

    if ticket.customer_email.split("@")[1] in settings.vip_domains:
        priority = priority.bump()
        run.log(f"cliente VIP: prioridade elevada para {priority}")
    if d.sentiment == "angry" and priority.rank > 2:          # irritado nunca fica abaixo de P2
        priority = priority.bump()
        run.log(f"cliente irritado: prioridade elevada para {priority}")
    if d.confidence < settings.min_confidence:
        team, needs_review = Team.HUMAN_TRIAGE, True
        run.log(f"confiança {d.confidence:.2f} < {settings.min_confidence}: triagem humana")
    return team, priority, needs_review

_seq = iter(range(1, 10_000))

def triage(llm, raw_body: str) -> tuple[int, str, TriageRun | None]:
    """Simula o handler HTTP: devolve (status, corpo JSON, execução)."""
    try:
        ticket = TicketIn.model_validate_json(raw_body)
    except ValidationError as e:
        return 422, json.dumps([{"field": ".".join(map(str, x["loc"])), "msg": x["msg"]} for x in e.errors(include_url=False)],
                               ensure_ascii=False), None
    run = TriageRun(ticket_id=f"T-{next(_seq):05d}")
    decision = call_llm_structured(llm, ticket, run)
    team, priority, review = apply_business_rules(ticket, decision, run)
    out = TicketOut(ticket_id=run.ticket_id, team=team, priority=priority, intent=decision.intent,
                    summary=decision.summary, needs_human_review=review, order_id=decision.order_id,
                    llm_attempts=len(run.calls))
    return 201, out.model_dump_json(), run

## 6. Rodando

Quatro tickets, com um LLM roteirizado:
1. Cobrança simples de cliente VIP.
2. Bug com cliente irritado: o LLM erra na primeira (prioridade fora do enum e `order_id` faltando), acerta após feedback, e a regra de sentimento sobe P3 → P2.
3. Pedido de feature com baixa confiança → triagem humana.
4. Corpo inválido → 422 sem gastar chamada de LLM.

In [7]:
tickets = [
    {"customer_email": "joana@bigcorp.com", "subject": "Fatura duplicada",
     "body": "Recebi duas cobranças do pedido 88231 neste mês. Podem estornar uma?"},
    {"customer_email": "rui@gmail.com", "subject": "App fora do ar!!!",
     "body": "Desde ontem o checkout dá erro 500 e não consigo finalizar o pedido 5521. Isso é um absurdo."},
    {"customer_email": "lia@startup.dev", "subject": "Ideia",
     "body": "Seria legal ter modo escuro e talvez integração com o calendário, não sei."},
    {"customer_email": "não-é-email", "subject": "x", "body": "curto", "urgent": True},
]

llm = ScriptedLLM([
    '{"intent": "billing", "priority": "P3", "confidence": 0.94, "summary": "Cobrança duplicada do pedido 88231; pede estorno.", '
    '"order_id": "88231", "sentiment": "calm"}',
    # tentativa inválida: prioridade "CRITICAL" e sem order_id
    'Triagem: {"intent": "bug", "priority": "CRITICAL", "confidence": 0.9, "summary": "Checkout com erro 500 impede compras.", "sentiment": "angry"}',
    '{"intent": "bug", "priority": "P3", "confidence": 0.9, "summary": "Checkout com erro 500 impede finalizar o pedido 5521.", '
    '"order_id": "5521", "sentiment": "angry"}',
    '{"intent": "feature", "priority": "P4", "confidence": 0.6, "summary": "Sugere modo escuro e integração com calendário.", '
    '"order_id": null, "sentiment": "calm"}',
])

runs = []
for t in tickets:
    status, body, run = triage(llm, json.dumps(t, ensure_ascii=False))
    print(f"\n[{status}] {body}")
    if run:
        runs.append(run)
        for ev in run.events:
            print("   ·", ev)


[201] {"ticket_id":"T-00001","team":"finance","priority":"P2","intent":"billing","summary":"Cobrança duplicada do pedido 88231; pede estorno.","needs_human_review":false,"order_id":"88231","llm_attempts":1}
   · cliente VIP: prioridade elevada para P2

[201] {"ticket_id":"T-00002","team":"engineering","priority":"P2","intent":"bug","summary":"Checkout com erro 500 impede finalizar o pedido 5521.","needs_human_review":false,"order_id":"5521","llm_attempts":2}
   · tentativa 1 inválida: priority: Input should be 'P1', 'P2', 'P3' or 'P4'; order_id: Field required
   · cliente irritado: prioridade elevada para P2

[201] {"ticket_id":"T-00003","team":"human_triage","priority":"P4","intent":"feature","summary":"Sugere modo escuro e integração com calendário.","needs_human_review":true,"order_id":null,"llm_attempts":1}
   · confiança 0.60 < 0.75: triagem humana

[422] [{"field": "urgent", "msg": "Extra inputs are not permitted"}, {"field": "customer_email", "msg": "value is not a valid email

## 7. Observabilidade da execução

In [8]:
print(f"{'ticket':<9}{'tentativas':>11}{'tokens':>8}{'erros':>7}")
for r in runs:
    print(f"{r.ticket_id:<9}{len(r.calls):>11}{r.tokens:>8}{sum(not c.ok for c in r.calls):>7}")
print("\nchamadas totais ao LLM:", len(llm.calls), "| tickets inválidos não chamaram o LLM")
print("taxa de primeira tentativa válida:", f"{sum(r.calls[0].ok for r in runs) / len(runs):.0%}")

ticket    tentativas  tokens  erros
T-00001            1     437      0
T-00002            2     942      1
T-00003            1     435      0

chamadas totais ao LLM: 4 | tickets inválidos não chamaram o LLM
taxa de primeira tentativa válida: 67%


## 8. Expondo como API (FastAPI)

O mesmo pipeline atrás de um endpoint. O FastAPI usa `TicketIn` para validar (422 automático) e `TicketOut` como `response_model`.

In [9]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI(title="Ticket triage")
api_llm = ScriptedLLM(['{"intent": "account", "priority": "P2", "confidence": 0.88, '
                       '"summary": "Cliente não consegue redefinir a senha.", "order_id": null, "sentiment": "frustrated"}'])

@app.post("/tickets", response_model=TicketOut, status_code=201)
def create_ticket(ticket: TicketIn):
    _, body, _ = triage(api_llm, ticket.model_dump_json())
    return TicketOut.model_validate_json(body)

client = TestClient(app)
r = client.post("/tickets", json={"customer_email": "ana@x.com", "subject": "Senha",
                                  "body": "O link de redefinir senha expira antes de eu clicar."})
print(r.status_code, r.json())
r = client.post("/tickets", json={"customer_email": "ana@x.com", "subject": "Senha"})
print(r.status_code, [(d["loc"], d["type"]) for d in r.json()["detail"]])
print("enum no OpenAPI:", app.openapi()["components"]["schemas"]["Team"]["enum"])

201 {'ticket_id': 'T-00004', 'team': 'support', 'priority': 'P2', 'intent': 'account', 'summary': 'Cliente não consegue redefinir a senha.', 'needs_human_review': False, 'order_id': None, 'llm_attempts': 1}
422 [(['body', 'body'], 'missing')]
enum no OpenAPI: ['finance', 'engineering', 'support', 'product', 'human_triage']


## O que levar deste pipeline

* **Pydantic nas três bordas** (config, entrada HTTP, saída do LLM) e na resposta — cada fronteira tem contrato e erro claro.
* **Enum como vocabulário único**: o mesmo `Intent` gera o prompt, restringe o schema, roteia (`ROUTING`) e aparece no OpenAPI.
* **dataclass no meio**: o `TriageRun` acumula tentativas, tokens e eventos sem custo de validação, e vira métrica/log no final.
* **Regras de negócio fora do LLM**: VIP, sentimento e limiar de confiança são código determinístico e testável —
  o LLM só classifica e resume.
* **Falhar barato**: entrada inválida é rejeitada antes de gastar tokens.